# Maximum Likelihood Estimation

In this section we will be exploring maximum likelihood estimation (MLE) across various distributions. 

We will start by deriving the MLE for the exponential distribution. The parameter we want to estimate is $\lambda$. Using standard calculus techniques we find that:

$$
\hat{\lambda} = \frac{1}{\bar{x}}
$$

Next we derive the MLE for the normal distribution parameters $\mu$ and $\sigma$. We arrive at:

$$
\hat{\mu} = \text{average} = \bar{x}
$$

$$
\hat{\sigma} = \sqrt{\frac{\sum_{i=1}^n (x_i - \mu)^2}{n}}
$$

Now that we have calculated these estimators, let's confirm that they accurately estimate the true parameter and show where the likelihood is maximized. To do so, we can graph the likelihood and log-likelihood functions of each and the corresponding MLE.

In [ ]:
import micropip

await micropip.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive, fixed

In [ ]:
def update_plot(sample_size, lambda_val):
    # we use the Generator API as it is a modern version of RandomState, constructing a new generator is done in the following line
    rng = np.random.default_rng()

    # we use the generator to create an exponential distribution sample
    data_exp = rng.exponential(scale=1 / lambda_val, size=sample_size)

    lambdas = np.linspace(0.01, 1.5, 500)
    sum_x = data_exp.sum()

    # likelihood for lambda
    likelihood = (lambdas**sample_size) * np.exp(-lambdas * sum_x)

    # log likelihood for lambda
    logL = np.log(likelihood)

    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(lambdas, likelihood)
    plt.axvline(
        sample_size / sum_x,
        color="C1",
        ls="--",
        label=f"MLE $\lambda$={sample_size/sum_x:.2f}",
    )
    plt.axvline(
        lambda_val,
        color="tab:olive",
        ls="--",
        label=f"True value $\lambda$={lambda_val:.2f}",
    )
    plt.title("Exponential Likelihood")
    plt.xlabel("$\lambda$")
    plt.ylabel("$L(\lambda)$")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(lambdas, logL)
    plt.axvline(
        sample_size / sum_x,
        color="C1",
        ls="--",
        label=f"MLE $\lambda$={sample_size/sum_x:.2f}",
    )
    plt.axvline(
        lambda_val,
        color="tab:olive",
        ls="--",
        label=f"True value $\lambda$={lambda_val:.2f}",
    )
    plt.title("Exponential Log-Likelihood")
    plt.xlabel("$\lambda$")
    plt.ylabel("$ln L(\lambda)$")
    plt.legend()
    plt.show()


lambda_true = 0.5
slider_n = widgets.IntSlider(
    value=50, min=10, max=200, step=10, description="Sample Size"
)
interactive_plot = interactive(
    update_plot, sample_size=slider_n, lambda_val=fixed(lambda_true)
)
interactive_plot

In [ ]:
def update_plot_mu(n, mu_val):
    rng = np.random.default_rng()
    data_norm = rng.normal(loc=mu_val, scale=2.0, size=n)
    mu = np.linspace(-2, 5, 200)

    likelihood = []
    logL_mu = []

    for m in mu:
        # mle of sigma
        sigma0 = np.sqrt(np.mean((data_norm - m) ** 2))

        log_likelihood = -n * np.log(sigma0 * np.sqrt(2 * np.pi)) - np.sum(
            (data_norm - m) ** 2
        ) / (2 * sigma0**2)
        logL_mu.append(log_likelihood)

        likelihood.append(np.exp(log_likelihood))

    likelihood = np.array(likelihood)
    logL_mu = np.array(logL_mu)

    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(mu, likelihood)
    plt.axvline(
        data_norm.mean(), color="r", ls="--", label=f"MLE $\mu$={data_norm.mean():.2f}"
    )
    plt.axvline(
        mu_val, color="tab:olive", ls="--", label=f"True value $\mu$={mu_val:.2f}"
    )
    plt.title("Likelihood vs $\mu$")
    plt.xlabel("$\mu$")
    plt.ylabel("$L(\mu)$")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(mu, logL_mu)
    plt.axvline(
        data_norm.mean(), color="r", ls="--", label=f"MLE $\mu$={data_norm.mean():.2f}"
    )
    plt.axvline(
        mu_val, color="tab:olive", ls="--", label=f"True value $\mu$={mu_val:.2f}"
    )
    plt.title("Log-Likelihood vs $\mu$")
    plt.xlabel("$\mu$")
    plt.ylabel("$\ln L(\mu)$")
    plt.legend()
    plt.show()


mu_true = 1.0
slider_n = widgets.IntSlider(
    value=50, min=10, max=250, step=10, description="Sample Size"
)
interactive_plot = interactive(update_plot_mu, n=slider_n, mu_val=fixed(mu_true))
interactive_plot

In [ ]:
def update_plot_sigma(n, sigma_val):
    rng = np.random.default_rng()
    data_norm = rng.normal(loc=1.0, scale=sigma_val, size=n)
    sigma = np.linspace(0.1, 5, 200)

    # mle of mu
    mu0 = data_norm.mean()
    likelihood = []
    logL_sigma = []

    sum_squared_deviations = np.sum((data_norm - mu0) ** 2)

    for s in sigma:
        log_likelihood = (
            -n / 2 * np.log(2 * np.pi)
            - n / 2 * np.log(s**2)
            - sum_squared_deviations / (2 * s**2)
        )
        logL_sigma.append(log_likelihood)

        likelihood.append(np.exp(log_likelihood))

    likelihood = np.array(likelihood)
    logL_sigma = np.array(logL_sigma)

    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(sigma, likelihood)
    plt.axvline(
        np.sqrt(np.sum((data_norm - mu0) ** 2) / len(data_norm)),
        color="r",
        ls="--",
        label=f"MLE $\sigma$={np.sqrt(((data_norm-mu0)**2).mean()):.2f}",
    )
    plt.axvline(
        sigma_val,
        color="tab:olive",
        ls="--",
        label=f"True value $\sigma$={sigma_val:.2f}",
    )
    plt.title("Likelihood vs $\sigma$")
    plt.xlabel("$\sigma$")
    plt.ylabel("$L(\sigma)$")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(sigma, logL_sigma)
    plt.axvline(
        np.sqrt(np.sum((data_norm - mu0) ** 2) / len(data_norm)),
        color="r",
        ls="--",
        label=f"MLE $\sigma$={np.sqrt(((data_norm-mu0)**2).mean()):.2f}",
    )
    plt.axvline(
        sigma_val,
        color="tab:olive",
        ls="--",
        label=f"True value $\sigma$={sigma_val:.2f}",
    )
    plt.title("Log-Likelihood vs $\sigma$")
    plt.xlabel("$\sigma$")
    plt.ylabel("$\ln L(\sigma)$")
    plt.legend()
    plt.show()


sigma_true = 2.0
slider_n = widgets.IntSlider(
    value=50, min=10, max=200, step=10, description="Sample Size"
)
interactive_plot = interactive(
    update_plot_sigma, n=slider_n, sigma_val=fixed(sigma_true)
)
interactive_plot

Run the code block below to visualise the surface with the likelihood as a function of both $\mu$ and $\sigma$.

In [ ]:
# add code for 3d plot

```{note} 
When calculating MLE we do not consider the value of the likelihood and the consequence of this value, only the fact that it is the maximum. This analysis of the value of the likelihood is out of the scope of the course.


An important property possessed by the MLE is that it is functionally invariant. This property stipulates that if you have some MLE $\hat{\theta}$ for a parameter $\theta$ and wish to calculate the MLE on a transformation $f(\theta)$, the MLE for $\alpha=f(\theta)$ is $\hat{\alpha}=f(\hat{\theta})$. 

To illustrate this property let us consider again the exponential distribution. We would like to compute the MLE for the expectation of the exponential distribution, $\mathbb{E}[X]$. 

$$
\begin{aligned}
\mathbb{E}[X] &= \frac 1 \lambda \\
\hat{ \mathbb{E}}[X] &= \frac 1 {\hat{\lambda}} \\
\hat{ \mathbb{E}}[X] &= \frac 1 {\frac{n}{\sum_{i=1}^n x_i}} \\
\hat{ \mathbb{E}}[X] &= \frac{\sum_{i=1}^n x_i}{n} \\
\end{aligned}
$$

In the code below we graph the likelihood function for this estimator to confirm that the MLE of the expectation of the exponential distribution is equal to the reciprocal of the MLE for $\lambda$. 

In [ ]:
def update_plot_exp(sample_size, lambda_val):

    rng = np.random.default_rng()
    data_exp = rng.exponential(scale=1 / lambda_val, size=sample_size)

    lambdas = np.linspace(0.1, 10, 500)
    n = data_exp.size
    sum_x = data_exp.sum()

    # likelihood for lambda
    likelihood = ((1.0 / lambdas) ** n) * np.exp(-sum_x / lambdas)

    # log likelihood for lambda
    logL = -n * np.log(lambdas) - sum_x / lambdas

    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(lambdas, likelihood)
    plt.axvline(sum_x / n, color="C1", ls="--", label=f"MLE $E[X]$={sum_x/n:.2f}")
    plt.axvline(
        1.0 / lambda_val,
        color="tab:olive",
        ls="--",
        label=f"True value $E[X]$={1.0/lambda_val:.2f}",
    )
    plt.title("Exponential Likelihood")
    plt.xlabel("$E[X]$")
    plt.ylabel("$L(E[X])$")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(lambdas, logL)
    plt.axvline(sum_x / n, color="C1", ls="--", label=f"MLE $E[X]$={sum_x/n:.2f}")
    plt.axvline(
        1.0 / lambda_val,
        color="tab:olive",
        ls="--",
        label=f"True value $E[X]={1.0/lambda_val:.2f}",
    )
    plt.title("Exponential Log-Likelihood")
    plt.xlabel("$E[X]$")
    plt.ylabel("$ln L(E[X])$")
    plt.legend()
    plt.show()


lambda_true = 0.5
slider_n = widgets.IntSlider(
    value=50, min=10, max=200, step=10, description="Sample Size"
)
interactive_plot = interactive(
    update_plot_exp, sample_size=slider_n, lambda_val=fixed(lambda_true)
)
interactive_plot